In [2]:
import importlib
import helper.common as common

importlib.reload(common)

dspy = common.get_dspy_instance()

## Class-based DSPy Signatures

For some advanced tasks, you need more verbose signatures. This is typically to:

- Clarify something about the nature of the task (expressed below as a *docstring*).
- Supply hints on the nature of an input field, expressed as a `desc` keyword argument for `dspy.InputField`.
- Supply constraints on an output field, expressed as a `desc` keyword argument for `dspy.OutputField`.

In [3]:
from typing import Literal

class RephraseWithDSPY(dspy.Signature):
    """Rephrase the text style based on user preference while preserving its meaning."""
    text: str = dspy.InputField()
    original_style: Literal["professional", "casual", "rude", "poetic", "angry"] = dspy.InputField()
    target_style: Literal["professional", "casual", "rude", "poetic", "angry"] = dspy.InputField()
    preserved_keywords: list[str] = dspy.OutputField()
    transformed_text: str = dspy.OutputField()
    update_metrics: dict[str, float] = dspy.OutputField(desc="Add score for formality, emotions, complexity")

#### Details

We’ve defined a `RephraseWithDSPY` signature class that inherits the `dspy.Signature`. This class will be used to rephrase a given sentence based on user preferred style, while preserving its original meaning and providing useful metrics.

In [4]:
text = "I have been playing pubg for 6 hours but my KD is not crossing 1, its feels like to break my mobile"

rephrase_text = dspy.Predict(RephraseWithDSPY)

response = rephrase_text(
    text=text,
    original_style="angry",
    target_style="poetic"
)


print("Rephrase String: ", response.transformed_text)
print("Preserved Keywords: ", response.preserved_keywords)
print("Update metrics: ", response.update_metrics)

Rephrase String:  For six long hours, in PUBG's digital realm I've toiled,
Yet my warrior's score, a K/D, remains uncoiled,
Refusing to ascend beyond the solitary mark.
A tempest rages within, a yearning dark,
To cast aside this device, its screen to rend,
And bring this frustrating struggle to an end.
Preserved Keywords:  ['pubg', '6 hours', 'KD', 'not crossing 1', 'break my mobile']
Update metrics:  {'formality': 0.8, 'emotions': 0.7, 'complexity': 0.8}


---
### DSPy Signature with Image Input - Multi-modal image classification

In [5]:
class DogPictureSignature(dspy.Signature):
    """Output the dog breed of the dog in the image."""
    image_1: dspy.Image = dspy.InputField(desc="An image of a dog")
    answer: str = dspy.OutputField(desc="The dog breed of the dog in the image")

image_url = "https://picsum.photos/id/237/200/300"
classify = dspy.Predict(DogPictureSignature)
response = classify(image_1=dspy.Image.from_url(image_url))

print("It belongs to the ", response.answer, " dog breed")

It belongs to the  Labrador Retriever  dog breed


--- 
## Type Resolution in Signatures

DSPy signatures support various annotation types:

* Basic types like `str`, `int`, `bool`
* Typing module types like `list[str]`, `dict[str, int]`, `Optional[float]`. `Union[str, int]`
* Custom types defined in your code
* Dot notation for nested types with proper configuration
Special data types like `dspy.Image`, `dspy.History`


---
### Create Custom Signature Types

In [7]:
import pydantic

# Simple custom type
class QueryResult(pydantic.BaseModel):
    text: str
    score: float

signature = dspy.Signature("query: str -> result: QueryResult")